# total_GVA engineered features

Rebuild the 10 CAAFE-selected engineered features for 'log_total_GVA_2023' from the raw 'features.csv' columns, save the dataset (keys + features + target), then split the data by 'is_swindon' and fit a TabPFN model to predict GVA for Swindon.

In [1]:
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings("ignore")
features = pd.read_csv("../data preprocessing+EDA/features.csv", low_memory=False)
print("features", features.shape)

features (1129, 83)


In [2]:
L4 = "Level 4 qualifications or above: degree (BA, BSc), higher degree (MA, PhD, PGCE), NVQ level 4 to 5, HNC, HND, RSA Higher Diploma, BTEC Higher level, professional qualifications (for example, teaching, nursing, accountancy) %"

src_cols = [
    "VOA_total_RV_million_2023",
    "Working_Age_Pop",
    "LU_total_2025_msoa",
    "Total_Pop_Mid2024",
    L4,
    "No qualifications %",
    "LU_diversity_1-HHI_2025",
    "employment_rate_per_pop",
    "full_to_part_ratio",
    "share_enterprises_kibs_2025_msoa",
    "VOA_total_RV_pct_change",
]

ids = features[["LSOA21CD", "MSOA21CD"]].copy()
df = features[src_cols].copy()
print("selected source columns", df.shape)

selected source columns (1129, 11)


In [3]:
df["log_voa_rv_2023"] = np.log1p(df["VOA_total_RV_million_2023"])
df["rv_per_working_age"] = df["VOA_total_RV_million_2023"] / (df["Working_Age_Pop"] + 1)
df["sme_density"] = df["LU_total_2025_msoa"] / (df["Total_Pop_Mid2024"] / 1000)
df["qualification_index"] = df[L4] - df["No qualifications %"]
df["firm_size_diversity"] = df["LU_diversity_1-HHI_2025"]
df["employment_rate"] = df["employment_rate_per_pop"]
df["share_kibs"] = df["share_enterprises_kibs_2025_msoa"]
df["rv_pct_change"] = df["VOA_total_RV_pct_change"]

df["rv_per_employee"] = df["rv_per_working_age"] / (df["employment_rate"] + 1e-6)
df["sme_qual_interaction"] = df["sme_density"] * df["qualification_index"]
df["employment_quality"] = df["employment_rate"] * df["full_to_part_ratio"]
df["modern_sector_leverage"] = df["share_kibs"] * df["qualification_index"]
df["asset_growth_diversity"] = df["rv_pct_change"] * df["firm_size_diversity"]

In [4]:
final_features = [
    "log_voa_rv_2023",
    "rv_per_working_age",
    "sme_density",
    "qualification_index",
    "firm_size_diversity",
    "rv_per_employee",
    "sme_qual_interaction",
    "employment_quality",
    "modern_sector_leverage",
    "asset_growth_diversity",
]

X = df[final_features].copy()
X = X.fillna(X.median())
print("final feature set", X.shape)
X.head()

final feature set (1129, 10)


,log_voa_rv_2023,rv_per_working_age,sme_density,qualification_index,firm_size_diversity,rv_per_employee,sme_qual_interaction,employment_quality,modern_sector_leverage,asset_growth_diversity
0,11.833714,90.671356,127.416520,19.683656,0.896111,2579.526688,2508.022893,0.021090,9.021675,8.157646
1,16.268274,8704.430255,408.819476,13.321084,0.921664,1848.735672,5445.918610,1.324213,5.398086,2.748291
2,14.707645,1371.742970,170.408982,-0.120289,0.905896,1954.926910,-20.498273,0.116947,-0.034923,11.903836
3,13.536891,587.142758,234.936429,3.593145,0.905896,4248.534257,844.160745,0.138198,1.043171,14.288140
4,14.753778,2255.558815,205.900430,11.370621,0.904844,8155.068718,2341.215710,0.138291,4.625337,4.415880


In [5]:
target = "log_total_GVA_2023"
upd = pd.concat([pd.read_csv("../data preprocessing+EDA/train_updated.csv"),
                 pd.read_csv("../data preprocessing+EDA/test_updated.csv")], ignore_index=True)

dataset = pd.concat([ids, X], axis=1).merge(upd[["LSOA21CD", target, "is_swindon"]], on="LSOA21CD", how="left")
dataset = dataset.dropna(subset=[target, "MSOA21CD"]).reset_index(drop=True)
dataset.to_csv("total_gva_engineered_features.csv", index=False, encoding="utf-8")
print("saved total_gva_engineered_features.csv", dataset.shape)

saved total_gva_engineered_features.csv (1125, 14)


## TabPFN model

In [6]:
from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score

data = dataset.dropna(subset=[target, "MSOA21CD"]).reset_index(drop=True)
X_mat = data[final_features].values
y = data[target].values
groups = data["MSOA21CD"].values
print("modelling rows", X_mat.shape)

modelling rows (1125, 10)


In [7]:
gkf = GroupKFold(n_splits=5)
pred = np.empty(len(y))
for tr, va in gkf.split(X_mat, y, groups):
    reg = TabPFNRegressor.create_default_for_version(ModelVersion.V3)
    reg.fit(X_mat[tr], y[tr])
    pred[va] = reg.predict(X_mat[va])

print(f"target: {target} (GroupKFold OOF)")
print(f"MAE = {mean_absolute_error(y, pred):.4f}")
print(f"RMSE = {np.sqrt(mean_squared_error(y, pred)):.4f}")
print(f"MAPE = {mean_absolute_percentage_error(y, pred):.4f}")
print(f"R2 = {r2_score(y, pred):.4f}")

target: log_total_GVA_2023 (GroupKFold OOF)
MAE = 0.3875
RMSE = 0.6348
MAPE = 0.1056
R2 = 0.6798


## Split by is_swindon

Train on the rest of the country 'Others' and test on 'Swindon' only.

In [8]:
train = data[data["is_swindon"] == "Others"].reset_index(drop=True)
test  = data[data["is_swindon"] == "Swindon"].reset_index(drop=True)

X_tr, X_te = train[final_features].values, test[final_features].values
y_tr, y_te = train[target].values, test[target].values
print("train(Others)", X_tr.shape, "test(Swindon)", X_te.shape)

train(Others) (988, 10) test(Swindon) (137, 10)


In [9]:
# Swindon result
reg = TabPFNRegressor.create_default_for_version(ModelVersion.V3)
reg.fit(X_tr, y_tr)
pred = reg.predict(X_te)

print(f"target: {target} (train=Others, test=Swindon)")
print(f"MAE = {mean_absolute_error(y_te, pred):.4f}")
print(f"RMSE = {np.sqrt(mean_squared_error(y_te, pred)):.4f}")
print(f"MAPE = {mean_absolute_percentage_error(y_te, pred):.4f}")
print(f"R2 = {r2_score(y_te, pred):.4f}")

target: log_total_GVA_2023 (train=Others, test=Swindon)
MAE = 0.4469
RMSE = 0.7137
MAPE = 0.1237
R2 = 0.6935


## XGBoost model

Same 10 features and the same two evaluations (GroupKFold OOF + is_swindon split), using XGBoost for comparison and for SHAP-based attribution downstream.

In [10]:
from xgboost import XGBRegressor

gkf = GroupKFold(n_splits=5)
pred = np.empty(len(y))
for tr, va in gkf.split(X_mat, y, groups):
    xgb = XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=4,
                       subsample=0.8, colsample_bytree=0.8, random_state=42)
    xgb.fit(X_mat[tr], y[tr])
    pred[va] = xgb.predict(X_mat[va])

print(f"target: {target} (XGBoost GroupKFold OOF)")
print(f"MAE = {mean_absolute_error(y, pred):.4f}")
print(f"RMSE = {np.sqrt(mean_squared_error(y, pred)):.4f}")
print(f"MAPE = {mean_absolute_percentage_error(y, pred):.4f}")
print(f"R2 = {r2_score(y, pred):.4f}")

target: log_total_GVA_2023 (XGBoost GroupKFold OOF)
MAE = 0.4792
RMSE = 0.7074
MAPE = 0.1331
R2 = 0.6024


In [11]:
# Swindon
xgb = XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=4,
                   subsample=0.8, colsample_bytree=0.8, random_state=42)
xgb.fit(X_tr, y_tr)
pred = xgb.predict(X_te)

print(f"target: {target} (XGBoost train=Others, test=Swindon)")
print(f"MAE = {mean_absolute_error(y_te, pred):.4f}")
print(f"RMSE = {np.sqrt(mean_squared_error(y_te, pred)):.4f}")
print(f"MAPE = {mean_absolute_percentage_error(y_te, pred):.4f}")
print(f"R2 = {r2_score(y_te, pred):.4f}")

target: log_total_GVA_2023 (XGBoost train=Others, test=Swindon)
MAE = 0.5006
RMSE = 0.7215
MAPE = 0.1396
R2 = 0.6868
